# Phase 4 - Hyperparameter Tuning: Extensive LightGBM Optimization
 
 This notebook performs comprehensive hyperparameter tuning on LightGBM to improve precision from 46% to 75-80% while maintaining 100% recall.
 
**Objective**: Optimize LightGBM for forensic timestomping detection
 
**Approach**:
- Stage 1: Randomized search across broad parameter space (150 combinations)
- Stage 2: Grid search refinement around best parameters
- Focus: Maximize F1 score (balance precision and recall)
- Validation: 5-fold stratified cross-validation
 
**Target Performance**:
- Recall: 100% (maintain perfect detection)
- Precision: 75-80% (improve from 46%)
- F1-Score: 0.86-0.89 (improve from 0.63)


In [9]:
# Cell 1: Import Libraries and Configuration
import pandas as pd
import numpy as np
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV, StratifiedKFold
from lightgbm import LGBMClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_score, recall_score, make_scorer
import warnings
import time
import pickle
import json
from scipy.stats import randint, uniform
warnings.filterwarnings('ignore')

# Paths
TRAINING_DATA_PATH = '/Users/soni/Github/Digital-Detectives_Thesis/data/processed/Phase 2 - Features/all_cases_combined_features.csv'
BASE_MODEL_PATH = '/Users/soni/Github/Digital-Detectives_Thesis/data/processed/Phase 3 - Model Training/lightgbm_model.pkl'
OUTPUT_DIR = '/Users/soni/Github/Digital-Detectives_Thesis/data/processed/Phase 4 - Hyperparameter Tuning/'

# Configuration
RANDOM_STATE = 42
CV_FOLDS = 5
N_ITER_RANDOM = 150  # Randomized search iterations
N_JOBS = -1  # Use all CPU cores

print("="*80)
print("EXTENSIVE LIGHTGBM HYPERPARAMETER TUNING")
print("="*80)
print(f"\nConfiguration:")
print(f"  Training Data: {TRAINING_DATA_PATH}")
print(f"  Random State: {RANDOM_STATE}")
print(f"  CV Folds: {CV_FOLDS}")
print(f"  Randomized Search Iterations: {N_ITER_RANDOM}")
print(f"  Parallel Jobs: {N_JOBS}")


EXTENSIVE LIGHTGBM HYPERPARAMETER TUNING

Configuration:
  Training Data: /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Phase 2 - Features/all_cases_combined_features.csv
  Random State: 42
  CV Folds: 5
  Randomized Search Iterations: 150
  Parallel Jobs: -1


In [10]:
# Cell 2: Load and Prepare Training Data
print("="*80)
print("LOADING TRAINING DATA")
print("="*80)

# Load training data
df = pd.read_csv(TRAINING_DATA_PATH)

print(f"Dataset loaded successfully")
print(f"Total samples: {len(df):,}")
print(f"Total columns (before cleanup): {len(df.columns)}")

# REMOVE LABEL LEAKAGE FEATURES
leakage_features = [
    'is_flagged_suspicious',
    'has_logfile_suspicious', 
    'has_usnjrnl_suspicious',
    'cross_artifact_detected'
]

print(f"\nRemoving {len(leakage_features)} label leakage features:")
for feat in leakage_features:
    if feat in df.columns:
        print(f"  - {feat}")

df = df.drop(leakage_features, axis=1, errors='ignore')
print(f"Total columns (after cleanup): {len(df.columns)}")

# Convert boolean columns to int
bool_cols = df.select_dtypes(include=['bool']).columns
if len(bool_cols) > 0:
    df[bool_cols] = df[bool_cols].astype(int)
    print(f"Converted {len(bool_cols)} boolean columns to int")

# Select only numeric columns for features
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

# Remove label from features
if 'ground_truth_label' in numeric_cols:
    numeric_cols.remove('ground_truth_label')

# Create feature matrix and labels
X = df[numeric_cols]
y = df['ground_truth_label']

print(f"\nFeature columns: {len(numeric_cols)} (should be 30)")
print(f"Suspicious files: {(y==1).sum():,} ({(y==1).sum()/len(y)*100:.2f}%)")
print(f"Benign files: {(y==0).sum():,} ({(y==0).sum()/len(y)*100:.2f}%)")

# Calculate class imbalance
class_weight_ratio = (y==0).sum() / (y==1).sum()
print(f"Class imbalance ratio: {class_weight_ratio:.2f}:1 (benign:suspicious)")

# Save feature names for later reference
feature_names = numeric_cols
print(f"\nFeatures saved: {len(feature_names)} forensic indicators")


LOADING TRAINING DATA
Dataset loaded successfully
Total samples: 88,190
Total columns (before cleanup): 45

Removing 4 label leakage features:
  - is_flagged_suspicious
  - has_logfile_suspicious
  - has_usnjrnl_suspicious
  - cross_artifact_detected
Total columns (after cleanup): 41

Feature columns: 30 (should be 30)
Suspicious files: 266 (0.30%)
Benign files: 87,924 (99.70%)
Class imbalance ratio: 330.54:1 (benign:suspicious)

Features saved: 30 forensic indicators


In [11]:
# Cell 3: Load Base Model Performance 
print("="*80)
print("BASE MODEL PERFORMANCE (PHASE 3)")
print("="*80)

# Load base model
try:
    with open(BASE_MODEL_PATH, 'rb') as f:
        base_model = pickle.load(f)
    
    # Evaluate base model on training data
    y_pred_base = base_model.predict(X)
    base_recall = recall_score(y, y_pred_base)
    base_precision = precision_score(y, y_pred_base)
    base_f1 = f1_score(y, y_pred_base)
    
    print("\nBase LightGBM Performance:")
    print(f"  Recall:    {base_recall:.4f} ({(y_pred_base[y==1] == 1).sum()}/{(y==1).sum()})")
    print(f"  Precision: {base_precision:.4f}")
    print(f"  F1-Score:  {base_f1:.4f}")
    
    print("\nBase Model Parameters:")
    for param, value in base_model.get_params().items():
        if param in ['n_estimators', 'num_leaves', 'max_depth', 'learning_rate', 
                     'min_child_samples', 'subsample', 'colsample_bytree', 'reg_alpha', 'reg_lambda']:
            print(f"  {param}: {value}")
    
except Exception as e:
    print(f"\nWarning: Could not load base model ({type(e).__name__})")
    print("Reason: Feature mismatch - Phase 3 model trained with label leakage features")
    print("\nUsing benchmark values from Phase 3 (without leakage features):")
    base_recall = 1.0000
    base_precision = 0.4630
    base_f1 = 0.6320
    base_model = None
    print(f"  Recall:    {base_recall:.4f}")
    print(f"  Precision: {base_precision:.4f}")
    print(f"  F1-Score:  {base_f1:.4f}")

print("\n" + "="*80)
print("TUNING GOAL")
print("="*80)
print("\nTarget Performance:")
print(f"  Recall:    ≥ 1.0000 (maintain perfect detection)")
print(f"  Precision: 0.7500-0.8000 (improve from {base_precision:.4f})")
print(f"  F1-Score:  0.8571-0.8889 (improve from {base_f1:.4f})")


BASE MODEL PERFORMANCE (PHASE 3)

Reason: Feature mismatch - Phase 3 model trained with label leakage features

Using benchmark values from Phase 3 (without leakage features):
  Recall:    1.0000
  Precision: 0.4630
  F1-Score:  0.6320

TUNING GOAL

Target Performance:
  Recall:    ≥ 1.0000 (maintain perfect detection)
  Precision: 0.7500-0.8000 (improve from 0.4630)
  F1-Score:  0.8571-0.8889 (improve from 0.6320)


[LightGBM] [Fatal] The number of features in data (30) is not the same as it was in training data (31).
You can set ``predict_disable_shape_check=true`` to discard this error, but please be aware what you are doing.


In [12]:
# Cell 4: Define Extensive Parameter Space 
print("="*80)
print("STAGE 1: RANDOMIZED SEARCH - BROAD PARAMETER EXPLORATION")
print("="*80)

# Define extensive parameter distributions for randomized search
param_distributions = {
    # Tree structure
    'num_leaves': randint(20, 150),           # Default: 31
    'max_depth': randint(5, 25),              # Default: -1 (no limit)
    
    # Learning parameters
    'learning_rate': uniform(0.001, 0.199),   # Range: 0.001 to 0.2
    'n_estimators': randint(50, 500),         # Default: 100
    
    # Sampling parameters
    'subsample': uniform(0.5, 0.5),           # Range: 0.5 to 1.0
    'colsample_bytree': uniform(0.5, 0.5),    # Range: 0.5 to 1.0
    
    # Regularization
    'reg_alpha': uniform(0, 10),              # L1 regularization
    'reg_lambda': uniform(0, 10),             # L2 regularization
    'min_child_samples': randint(5, 100),     # Default: 20
    
    # Class imbalance
    'scale_pos_weight': [class_weight_ratio],
    
    # Other important parameters
    'min_split_gain': uniform(0, 1),
    'min_child_weight': uniform(0.001, 0.1)
}

print("\nParameter Distributions:")
print(f"  num_leaves: 20-150")
print(f"  max_depth: 5-25")
print(f"  learning_rate: 0.001-0.2")
print(f"  n_estimators: 50-500")
print(f"  subsample: 0.5-1.0")
print(f"  colsample_bytree: 0.5-1.0")
print(f"  reg_alpha: 0-10")
print(f"  reg_lambda: 0-10")
print(f"  min_child_samples: 5-100")
print(f"  min_split_gain: 0-1")
print(f"  min_child_weight: 0.001-0.1")
print(f"  scale_pos_weight: {class_weight_ratio:.2f} (fixed)")

print(f"\nRandomized Search: {N_ITER_RANDOM} random combinations")
print(f"Total search space: Billions of combinations")
print(f"Estimated runtime: 15-30 minutes")


STAGE 1: RANDOMIZED SEARCH - BROAD PARAMETER EXPLORATION

Parameter Distributions:
  num_leaves: 20-150
  max_depth: 5-25
  learning_rate: 0.001-0.2
  n_estimators: 50-500
  subsample: 0.5-1.0
  colsample_bytree: 0.5-1.0
  reg_alpha: 0-10
  reg_lambda: 0-10
  min_child_samples: 5-100
  min_split_gain: 0-1
  min_child_weight: 0.001-0.1
  scale_pos_weight: 330.54 (fixed)

Randomized Search: 150 random combinations
Total search space: Billions of combinations
Estimated runtime: 15-30 minutes


In [13]:
# Cell 5: Execute Randomized Search
print("="*80)
print("EXECUTING RANDOMIZED SEARCH")
print("="*80)

start_time = time.time()

# Initialize base estimator
lgbm_base = LGBMClassifier(
    random_state=RANDOM_STATE,
    n_jobs=N_JOBS,
    verbose=-1,
    force_col_wise=True
)

# Create custom scorer that prioritizes recall
def custom_scorer(y_true, y_pred):
    recall = recall_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    
    # Penalize heavily if recall < 1.0
    if recall < 1.0:
        return recall * 0.5  # Heavy penalty
    else:
        return f1  # Optimize F1 when recall is perfect

scorer = make_scorer(custom_scorer)

# Randomized search
random_search = RandomizedSearchCV(
    estimator=lgbm_base,
    param_distributions=param_distributions,
    n_iter=N_ITER_RANDOM,
    cv=StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE),
    scoring=scorer,
    n_jobs=N_JOBS,
    verbose=2,
    random_state=RANDOM_STATE,
    return_train_score=True
)

print(f"\nStarting randomized search with {N_ITER_RANDOM} iterations...")
print("This will take approximately 15-30 minutes...\n")

random_search.fit(X, y)

elapsed_random = time.time() - start_time

print(f"\n{'='*80}")
print(f"RANDOMIZED SEARCH COMPLETED in {elapsed_random/60:.2f} minutes")
print(f"{'='*80}")

# Best parameters from randomized search
print("\nBest Parameters from Randomized Search:")
for param, value in random_search.best_params_.items():
    print(f"  {param}: {value}")

# Evaluate best model from randomized search
y_pred_random = random_search.best_estimator_.predict(X)
random_recall = recall_score(y, y_pred_random)
random_precision = precision_score(y, y_pred_random)
random_f1 = f1_score(y, y_pred_random)

print(f"\nBest CV Score: {random_search.best_score_:.4f}")
print(f"\nTraining Set Performance:")
print(f"  Recall:    {random_recall:.4f} ({(y_pred_random[y==1] == 1).sum()}/{(y==1).sum()})")
print(f"  Precision: {random_precision:.4f}")
print(f"  F1-Score:  {random_f1:.4f}")

print(f"\nImprovement over Base Model:")
print(f"  Recall:    {random_recall - base_recall:+.4f}")
print(f"  Precision: {random_precision - base_precision:+.4f}")
print(f"  F1-Score:  {random_f1 - base_f1:+.4f}")

# Store randomized search results
random_results = {
    'best_params': random_search.best_params_,
    'best_cv_score': random_search.best_score_,
    'train_recall': random_recall,
    'train_precision': random_precision,
    'train_f1': random_f1,
    'time_minutes': elapsed_random/60
}


EXECUTING RANDOMIZED SEARCH

Starting randomized search with 150 iterations...
This will take approximately 15-30 minutes...

Fitting 5 folds for each of 150 candidates, totalling 750 fits
[CV] END colsample_bytree=0.6872700594236812, learning_rate=0.19019214697557332, max_depth=15, min_child_samples=76, min_child_weight=0.06086584841970366, min_split_gain=0.15601864044243652, n_estimators=264, num_leaves=94, reg_alpha=4.592488919658671, reg_lambda=3.337086111390218, scale_pos_weight=330.54135338345867, subsample=0.5714334089609704; total time=   1.3s
[CV] END colsample_bytree=0.6872700594236812, learning_rate=0.19019214697557332, max_depth=15, min_child_samples=76, min_child_weight=0.06086584841970366, min_split_gain=0.15601864044243652, n_estimators=264, num_leaves=94, reg_alpha=4.592488919658671, reg_lambda=3.337086111390218, scale_pos_weight=330.54135338345867, subsample=0.5714334089609704; total time=   1.5s
[CV] END colsample_bytree=0.6872700594236812, learning_rate=0.19019214697

In [14]:
# Cell 6: Refine with Grid Search
print("="*80)
print("STAGE 2: GRID SEARCH - FINE-TUNING AROUND BEST PARAMETERS")
print("="*80)

# Extract best parameters from randomized search
best_params = random_search.best_params_

# Create refined parameter grid around best values
def create_refinement_grid(best_params):
    grid = {}
    
    # num_leaves: ±20 around best, but keep in valid range
    best_leaves = best_params['num_leaves']
    grid['num_leaves'] = [
        max(20, best_leaves - 20),
        max(20, best_leaves - 10),
        best_leaves,
        min(150, best_leaves + 10),
        min(150, best_leaves + 20)
    ]
    
    # learning_rate: ±0.02 around best
    best_lr = best_params['learning_rate']
    grid['learning_rate'] = [
        max(0.001, best_lr - 0.02),
        max(0.001, best_lr - 0.01),
        best_lr,
        min(0.2, best_lr + 0.01),
        min(0.2, best_lr + 0.02)
    ]
    
    # n_estimators: ±50 around best
    best_est = best_params['n_estimators']
    grid['n_estimators'] = [
        max(50, best_est - 50),
        max(50, best_est - 25),
        best_est,
        min(500, best_est + 25),
        min(500, best_est + 50)
    ]
    
    # max_depth: ±2 around best
    best_depth = best_params['max_depth']
    grid['max_depth'] = [
        max(5, best_depth - 2),
        max(5, best_depth - 1),
        best_depth,
        min(25, best_depth + 1),
        min(25, best_depth + 2)
    ]
    
    # Keep best values for other parameters (single value = fixed)
    grid['subsample'] = [best_params['subsample']]
    grid['colsample_bytree'] = [best_params['colsample_bytree']]
    grid['reg_alpha'] = [best_params['reg_alpha']]
    grid['reg_lambda'] = [best_params['reg_lambda']]
    grid['min_child_samples'] = [best_params['min_child_samples']]
    grid['scale_pos_weight'] = [best_params['scale_pos_weight']]
    grid['min_split_gain'] = [best_params['min_split_gain']]
    grid['min_child_weight'] = [best_params['min_child_weight']]
    
    return grid

refinement_grid = create_refinement_grid(best_params)

# Calculate total combinations
total_combinations = np.prod([len(v) for v in refinement_grid.values()])

print(f"\nRefinement Grid:")
print(f"  num_leaves: {refinement_grid['num_leaves']}")
print(f"  learning_rate: {[f'{x:.4f}' for x in refinement_grid['learning_rate']]}")
print(f"  n_estimators: {refinement_grid['n_estimators']}")
print(f"  max_depth: {refinement_grid['max_depth']}")
print(f"\nTotal combinations: {total_combinations}")
print(f"Estimated runtime: 10-20 minutes")

start_time_grid = time.time()

# Grid search
grid_search = GridSearchCV(
    estimator=lgbm_base,
    param_grid=refinement_grid,
    cv=StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE),
    scoring=scorer,
    n_jobs=N_JOBS,
    verbose=2,
    return_train_score=True
)

print(f"\nStarting grid search refinement...")
grid_search.fit(X, y)

elapsed_grid = time.time() - start_time_grid

print(f"\n{'='*80}")
print(f"GRID SEARCH COMPLETED in {elapsed_grid/60:.2f} minutes")
print(f"{'='*80}")

# Best parameters from grid search
print("\nFinal Optimized Parameters:")
for param, value in grid_search.best_params_.items():
    if isinstance(value, float):
        print(f"  {param}: {value:.4f}")
    else:
        print(f"  {param}: {value}")

# Evaluate final model
y_pred_final = grid_search.best_estimator_.predict(X)
final_recall = recall_score(y, y_pred_final)
final_precision = precision_score(y, y_pred_final)
final_f1 = f1_score(y, y_pred_final)

print(f"\nBest CV Score: {grid_search.best_score_:.4f}")
print(f"\nTraining Set Performance:")
print(f"  Recall:    {final_recall:.4f} ({(y_pred_final[y==1] == 1).sum()}/{(y==1).sum()})")
print(f"  Precision: {final_precision:.4f}")
print(f"  F1-Score:  {final_f1:.4f}")

# Store final results
final_results = {
    'best_params': grid_search.best_params_,
    'best_cv_score': grid_search.best_score_,
    'train_recall': final_recall,
    'train_precision': final_precision,
    'train_f1': final_f1,
    'time_minutes': elapsed_grid/60
}


STAGE 2: GRID SEARCH - FINE-TUNING AROUND BEST PARAMETERS

Refinement Grid:
  num_leaves: [94, 104, 114, 124, 134]
  learning_rate: ['0.1478', '0.1578', '0.1678', '0.1778', '0.1878']
  n_estimators: [196, 221, 246, 271, 296]
  max_depth: [5, 5, 6, 7, 8]

Total combinations: 625
Estimated runtime: 10-20 minutes

Starting grid search refinement...
Fitting 5 folds for each of 625 candidates, totalling 3125 fits
[CV] END colsample_bytree=0.9210594615678543, learning_rate=0.14782741223751644, max_depth=5, min_child_samples=5, min_child_weight=0.0023671964826997285, min_split_gain=0.07535906035246231, n_estimators=196, num_leaves=94, reg_alpha=7.499107494699718, reg_lambda=9.131657522576429, scale_pos_weight=330.54135338345867, subsample=0.792574766162736; total time=   1.0s
[CV] END colsample_bytree=0.9210594615678543, learning_rate=0.14782741223751644, max_depth=5, min_child_samples=5, min_child_weight=0.0023671964826997285, min_split_gain=0.07535906035246231, n_estimators=196, num_leaves=

In [15]:
# Cell 7: Comprehensive Performance Comparison 
print("="*80)
print("COMPREHENSIVE PERFORMANCE COMPARISON")
print("="*80)

# Create comparison table
comparison_data = {
    'Model': [
        'Base LightGBM (Phase 3)',
        'Randomized Search',
        'Grid Search (Final)'
    ],
    'Recall': [
        base_recall,
        random_recall,
        final_recall
    ],
    'Precision': [
        base_precision,
        random_precision,
        final_precision
    ],
    'F1-Score': [
        base_f1,
        random_f1,
        final_f1
    ],
    'Time (min)': [
        0.0,
        random_results['time_minutes'],
        final_results['time_minutes']
    ]
}

comparison_df = pd.DataFrame(comparison_data)

print("\n")
print(comparison_df.to_string(index=False))

# Calculate improvements
print("\n" + "="*80)
print("IMPROVEMENT ANALYSIS")
print("="*80)

recall_improvement = final_recall - base_recall
precision_improvement = final_precision - base_precision
f1_improvement = final_f1 - base_f1

print(f"\nFinal Model vs Base Model:")
print(f"  Recall:    {base_recall:.4f} → {final_recall:.4f} ({recall_improvement:+.4f}, {recall_improvement/base_recall*100:+.2f}%)")
print(f"  Precision: {base_precision:.4f} → {final_precision:.4f} ({precision_improvement:+.4f}, {precision_improvement/base_precision*100:+.2f}%)")
print(f"  F1-Score:  {base_f1:.4f} → {final_f1:.4f} ({f1_improvement:+.4f}, {f1_improvement/base_f1*100:+.2f}%)")

# Check if targets were met
print("\n" + "="*80)
print("TARGET ACHIEVEMENT")
print("="*80)

recall_target = 1.0
precision_target_min = 0.75
precision_target_max = 0.80
f1_target_min = 0.857

print(f"\nRecall Target: ≥ {recall_target:.4f}")
print(f"  Achieved: {final_recall:.4f} {'✓ PASS' if final_recall >= recall_target else '✗ FAIL'}")

print(f"\nPrecision Target: {precision_target_min:.4f} - {precision_target_max:.4f}")
print(f"  Achieved: {final_precision:.4f} {'✓ PASS' if precision_target_min <= final_precision <= precision_target_max else '✗ FAIL'}")

print(f"\nF1-Score Target: ≥ {f1_target_min:.4f}")
print(f"  Achieved: {final_f1:.4f} {'✓ PASS' if final_f1 >= f1_target_min else '✗ FAIL'}")

# Total tuning time
total_time = elapsed_random + elapsed_grid
print(f"\nTotal Tuning Time: {total_time/60:.2f} minutes")


COMPREHENSIVE PERFORMANCE COMPARISON


                  Model  Recall  Precision  F1-Score  Time (min)
Base LightGBM (Phase 3)     1.0   0.463000  0.632000    0.000000
      Randomized Search     1.0   0.422893  0.594413    3.500126
    Grid Search (Final)     1.0   0.428341  0.599775   69.156220

IMPROVEMENT ANALYSIS

Final Model vs Base Model:
  Recall:    1.0000 → 1.0000 (+0.0000, +0.00%)
  Precision: 0.4630 → 0.4283 (-0.0347, -7.49%)
  F1-Score:  0.6320 → 0.5998 (-0.0322, -5.10%)

TARGET ACHIEVEMENT

Recall Target: ≥ 1.0000
  Achieved: 1.0000 ✓ PASS

Precision Target: 0.7500 - 0.8000
  Achieved: 0.4283 ✗ FAIL

F1-Score Target: ≥ 0.8570
  Achieved: 0.5998 ✗ FAIL

Total Tuning Time: 72.66 minutes


In [16]:
# Cell 8: Feature Importance Analysis
print("="*80)
print("FEATURE IMPORTANCE ANALYSIS")
print("="*80)

# Get feature importance from final model
feature_importance = grid_search.best_estimator_.feature_importances_

# Create dataframe
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': feature_importance
}).sort_values('Importance', ascending=False)

print("\nTop 15 Most Important Features:")
print(importance_df.head(15).to_string(index=False))

print("\n" + "="*80)
print("KEY FORENSIC INDICATORS")
print("="*80)

# Highlight key timestomping features
key_features = [
    'zero_in_nanoseconds',
    'zero_in_nanoseconds_lf',
    'time_reversal_event',
    'basic_info_changed',
    'timestamp_changed_to_past',
    'cross_artifact_validation_score'
]

print("\nImportance of Key Timestomping Indicators:")
for feat in key_features:
    if feat in importance_df['Feature'].values:
        importance = importance_df[importance_df['Feature'] == feat]['Importance'].values[0]
        rank = importance_df[importance_df['Feature'] == feat].index[0] + 1
        print(f"  {feat:35s} Importance: {importance:8.4f}  Rank: #{rank}")

# Save feature importance
importance_df.to_csv(OUTPUT_DIR + 'feature_importance.csv', index=False)
print(f"\nFeature importance saved: {OUTPUT_DIR}feature_importance.csv")


FEATURE IMPORTANCE ANALYSIS

Top 15 Most Important Features:
                        Feature  Importance
                filename_length         110
                     path_depth          69
    multiple_timestamps_changed          64
 zero_in_nanoseconds_suspicious          46
              in_temp_directory          45
                  is_executable          40
             basic_info_changed          28
         modified_time_modified          25
         creation_time_modified          17
           same_as_another_file          13
         accessed_time_modified           7
cross_artifact_validation_score           6
               in_program_files           5
         zero_in_nanoseconds_lf           5
              mft_time_modified           4

KEY FORENSIC INDICATORS

Importance of Key Timestomping Indicators:
  zero_in_nanoseconds                 Importance:   0.0000  Rank: #3
  zero_in_nanoseconds_lf              Importance:   5.0000  Rank: #1
  time_reversal_event       

In [17]:
# Cell 9: Save Final Model and Results
print("="*80)
print("SAVING TUNED MODEL")
print("="*80)

# Save the final tuned model
tuned_model_path = OUTPUT_DIR + 'lightgbm_tuned_model.pkl'
with open(tuned_model_path, 'wb') as f:
    pickle.dump(grid_search.best_estimator_, f)

print(f"\nTuned model saved: {tuned_model_path}")

# Save tuning results
tuning_results = {
    'base_model': {
        'recall': float(base_recall),
        'precision': float(base_precision),
        'f1_score': float(base_f1)
    },
    'randomized_search': {
        'best_params': {k: (int(v) if isinstance(v, np.integer) else float(v) if isinstance(v, np.floating) else v) 
                       for k, v in random_results['best_params'].items()},
        'recall': float(random_recall),
        'precision': float(random_precision),
        'f1_score': float(random_f1),
        'cv_score': float(random_results['best_cv_score']),
        'time_minutes': float(random_results['time_minutes'])
    },
    'grid_search': {
        'best_params': {k: (int(v) if isinstance(v, np.integer) else float(v) if isinstance(v, np.floating) else v) 
                       for k, v in final_results['best_params'].items()},
        'recall': float(final_recall),
        'precision': float(final_precision),
        'f1_score': float(final_f1),
        'cv_score': float(final_results['best_cv_score']),
        'time_minutes': float(final_results['time_minutes'])
    },
    'improvement': {
        'recall': float(recall_improvement),
        'precision': float(precision_improvement),
        'f1_score': float(f1_improvement)
    },
    'total_tuning_time_minutes': float(total_time/60)
}

results_path = OUTPUT_DIR + 'tuning_results.json'
with open(results_path, 'w') as f:
    json.dump(tuning_results, f, indent=2)

print(f"Tuning results saved: {results_path}")

# Save comparison table
comparison_df.to_csv(OUTPUT_DIR + 'model_comparison.csv', index=False)
print(f"Comparison table saved: {OUTPUT_DIR}model_comparison.csv")


SAVING TUNED MODEL

Tuned model saved: /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Phase 4 - Hyperparameter Tuning/lightgbm_tuned_model.pkl
Tuning results saved: /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Phase 4 - Hyperparameter Tuning/tuning_results.json
Comparison table saved: /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Phase 4 - Hyperparameter Tuning/model_comparison.csv


In [19]:
print("="*80)
print("HYPERPARAMETER TUNING COMPLETE")
print("="*80)

print("\nSUMMARY:")
print(f"  Base Model F1:  {base_f1:.4f}")
print(f"  Tuned Model F1: {final_f1:.4f}")
print(f"  Improvement:    {f1_improvement:+.4f} ({f1_improvement/base_f1*100:+.2f}%)")
print(f"  Tuning Time:    {total_time/60:.2f} minutes")

print("\n" + "="*80)
print("NEXT STEPS: LONE WOLF VALIDATION")
print("="*80)

print("\nThe tuned model must now be validated on the Lone Wolf held-out test set.")
print("\nValidation Procedure:")
print("  1. Load Lone Wolf $LogFile and $UsnJrnl data")
print("  2. Apply same feature engineering pipeline")
print("  3. Load tuned model and generate predictions")
print("  4. Compare against ground truth (12 known timestomped files)")
print("  5. Calculate final performance metrics")
print("\nExpected Outcome:")
print("  - Recall: 12/12 files detected (100%)")
print(f"  - Precision: Improved from 50-62.5% baseline")
print("  - Confidence: All detections at 70%+ confidence")

print("\nFiles Ready for Validation:")
print(f"  Tuned Model: {tuned_model_path}")
print(f"  Feature Names: {len(feature_names)} features")
print(f"  Model Parameters: Saved in {results_path}")

HYPERPARAMETER TUNING COMPLETE

SUMMARY:
  Base Model F1:  0.6320
  Tuned Model F1: 0.5998
  Improvement:    -0.0322 (-5.10%)
  Tuning Time:    72.66 minutes

NEXT STEPS: LONE WOLF VALIDATION

The tuned model must now be validated on the Lone Wolf held-out test set.

Validation Procedure:
  1. Load Lone Wolf $LogFile and $UsnJrnl data
  2. Apply same feature engineering pipeline
  3. Load tuned model and generate predictions
  4. Compare against ground truth (12 known timestomped files)
  5. Calculate final performance metrics

Expected Outcome:
  - Recall: 12/12 files detected (100%)
  - Precision: Improved from 50-62.5% baseline
  - Confidence: All detections at 70%+ confidence

Files Ready for Validation:
  Tuned Model: /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Phase 4 - Hyperparameter Tuning/lightgbm_tuned_model.pkl
  Feature Names: 30 features
  Model Parameters: Saved in /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Phase 4 - Hyperparameter Tuning/